In [8]:
!pip install numpy
!pip install matplotlib
!pip install pandas
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 35.1 MB/s  0:00:006m0:00:01


In [9]:
import pandas as pd

cols = [
    "burst",        # Burst counter
    "ensemble",     # Ensemble counter (1-65536)
    "u",            # Velocity Beam1|X|East (m/s)
    "v",            # Velocity Beam2|Y|North (m/s)
    "w",            # Velocity Beam3|Z|Up (m/s)
    "amp1",         # Amplitude Beam1 (counts)
    "amp2",         # Amplitude Beam2 (counts)
    "amp3",         # Amplitude Beam3 (counts)
    "snr1",         # SNR Beam1 (dB)
    "snr2",         # SNR Beam2 (dB)
    "snr3",         # SNR Beam3 (dB)
    "corr1",        # Correlation Beam1 (%)
    "corr2",        # Correlation Beam2 (%)
    "corr3",        # Correlation Beam3 (%)
    "pressure",     # Pressure (dbar)
    "analog1",      # Analog input 1
    "analog2",      # Analog input 2
    "checksum",     # Checksum (1 = failed)
]


df = pd.read_csv("SSC01.dat", sep=r"\s+", header=None, names=cols)

In [10]:
df["t"] = df.index / 16
start = pd.Timestamp("2026-09-29 06:00:00")
df["time"] = start + pd.to_timedelta(df["t"], unit="s")

In [12]:
DAY = start.strftime("%Y-%m-%d")   


manoeuvres = {
    "A1-11": "07:40:22",
    "A1-12": "07:46:51",
    "A2-11": "07:57:17",
    "A2-12": "08:02:11",
    "A3-11": "08:07:25",
    "A3-12": "08:12:08",
    "B-11":  "08:17:53",
    "B-12":  "08:23:40",
    "C-11":  "08:40:40",
    "C-12":  "08:45:57",
    "E-11":  "08:51:35",
    "E-12":  "08:57:08",
    
    "A1-21": "09:02:35",
    "A1-22": "09:07:47",
    "A2-21": "09:19:14",
    "A2-22": "09:29:27",
    "A3-21": "09:29:27",
    "A3-22": "09:35:17",
    "B-21":  "09:42:33",
    "B-22":  "09:49:34",
    "C-21":  "09:55:04",
    "C-22":  "09:58:54",
    "E-21":  "10:02:51",
    "E-22":  "10:06:45",

    "A1-31": "10:15:16",
    "A2-31": "10:17:03",
    "A3-31": "10:19:05",
    "B-31":  "10:22:36",
    "C-31":  "10:25:25",
    "E-31":  "10:31:33",
    "D-31":  "10:37:55",
    "D-32":  "10:41:15",
    "D-33":  "10:44:33",
    "F-31":  "10:49:06", #time of 3rd pass 
    "F-32":  "10:53:57",
    "F-33":  "11:00:27",




    "A1-41": "07:52:21",
    "B-41":  "08:29:40",
    "B-42":  "08:36:05",
    "A2-41": "09:24:16",
    "E-41": "10:10:30",
    "C-41": "10:28:22",
    "A1-41": "10:34:17",
    "F-41": "11:05:41"  

}

def select(ids):
    """Accepts one id ("B-11"), a list (["B-11", "B-12"]) or a whole type ("B")."""
    if isinstance(ids, str):
        ids = [ids]
    out = []
    for i in ids:
        if i in manoeuvres:
            out.append(i)
        else:
            matches = [k for k in manoeuvres if k.split("-")[0] == i]
            if not matches:
                raise KeyError(f"Unknown manoeuvre or type: {i}")
            out += matches
    return out

def plot_overlay(data, ids, before=120, after=120, col="analog1"):
    fig, ax = plt.subplots(figsize=(10, 5))
    for name in select(ids):
        c = pd.Timestamp(f"{DAY} {manoeuvres[name]}")
        win = data[(data["time"] >= c - pd.Timedelta(seconds=before)) &
                   (data["time"] <= c + pd.Timedelta(seconds=after))]
        if win.empty:
            print(f"No data around {name} ({c:%H:%M:%S}), skipped")
            continue
        rel_t = (win["time"] - c).dt.total_seconds()
        ax.plot(rel_t, win[col], lw=0.8, label=f"{name}  ({c:%H:%M:%S})")
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.set_xlabel("Time relative to boat passage (s)")
    ax.set_ylabel(col)
    ax.set_title(f"{col}: {before} s before to {after} s after passage")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_overlay(df, "E")                       # one manoeuvre
#plot_overlay(df, ["B-11", "B-12"], col="analog")    # a few
#plot_overlay(df, "B", col="analog1")                 # every B manoeuvre
#plot_overlay(df, "A1")                         # every A1 manoeuvre

NameError: name 'plot_overlay' is not defined

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=cb8feeb3-afee-4b20-ab89-f0abd9348b44' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>